<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Unsupervised exploration

**[Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/)** · Machine Learning: From Problem to Reliable Model · Module 7, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** group customers with k-means, scale the columns first, draw the groups in two dimensions with PCA, use inertia and silhouette as hints for the number of groups, and ask a person who knows the business what the groups mean. The module practice then looks at the missed escalations and at the groups together.

| | |
|---|---|
| **Time** | About 70 minutes, with the module practice |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | Unsupervised learning, from [Unsupervised learning](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/). Scaling, from [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/). The four kinds of error and `slice_table()`, from [Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M07-L04-unsupervised-exploration/unsupervised-exploration-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. It is the final version from Select useful features. The module practice uses it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One row per customer

**Clustering** puts similar rows into groups without a target. Here, the rows are customers, because Grace can act on a group of customers (for example, give them one contact person). The next cell builds one row per customer from the **train** tickets only. Guests have no customer ID, so they are left out.

- `tickets`: the customer's tickets in the train period.
- `order_value`: the median order value of their tickets.
- `word_count`: the mean length of their first messages.
- `tenure_days`: how long they were a customer at their last train ticket.

> **Check.** You should see `847 customers; 202 guest tickets left out`, and a table where `tenure_days` has a mean of 1272.9 and `tickets` a maximum of 262.

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train = load("train")
known = train[train["customer_id"].notna()]
customers = known.groupby("customer_id").agg(
    tickets=("ticket_id", "size"), order_value=("order_value", "median"),
    word_count=("word_count", "mean"), tenure_days=("customer_tenure_days", "max"))
print(len(customers), "customers;", train["customer_id"].isna().sum(), "guest tickets left out")
customers.describe().round(1)

## 2. Distance needs scaling: six customers

k-means groups rows by distance. Six made-up customers have two columns: `tickets` (2 to 50) and `tenure_days` (300 to 2,400). A and E are both small customers. C is a big customer.

> **Predict.** As recorded, is A closer to C or to E? Then run the cell. The second line uses scaled columns: (value − mean) ÷ SD.

In [ ]:
tiny = pd.DataFrame({"tickets": [2, 3, 40, 45, 4, 50],
                     "tenure_days": [300, 2400, 350, 2300, 1300, 1250]}, index=list("ABCDEF"))


def distance(df, a, b):
    """The straight-line distance between two rows."""
    return round(float(((df.loc[a] - df.loc[b]) ** 2).sum() ** 0.5), 2)


scaled_tiny = (tiny - tiny.mean()) / tiny.std(ddof=0)
print("as recorded: A to C", distance(tiny, "A", "C"), "| A to E", distance(tiny, "A", "E"))
print("scaled:      A to C", distance(scaled_tiny, "A", "C"), "| A to E", distance(scaled_tiny, "A", "E"))

> **Check.** You should see `A to C 62.8 | A to E 1000.0` as recorded, and `A to C 1.79 | A to E 1.21` scaled.

As recorded, a difference of 1,000 days hides a difference of 38 tickets: the column with the larger numbers decides everything. Scaled, each column counts in SDs, and A is closer to E.

Now run k-means with 2 groups on both versions. Think of placing k ice-cream stands on a beach: each person walks to the nearest stand, then each stand moves to the middle of its people, again and again. **k-means** starts with k centres, puts each row in the group of its nearest centre, moves each centre to the mean of its group, and repeats until nothing changes.

In [ ]:
from sklearn.cluster import KMeans

for name, data in [("as recorded", tiny), ("scaled", scaled_tiny)]:
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(data)
    groups = [list(tiny.index[labels == label]) for label in sorted(set(labels), key=list(labels).index)]
    print(name, groups)

> **Check.** You should see `as recorded [['A', 'C', 'E', 'F'], ['B', 'D']]` and `scaled [['A', 'B', 'E'], ['C', 'D', 'F']]`.

As recorded, the groups are "short tenure" and "long tenure". Scaled, they are "few tickets" and "many tickets", because the tickets differ more, relative to their spread.

## 3. k-means on 847 customers

The next cell puts the imputer, the scaler and k-means in one pipeline, as in Module 4. `n_init=10` runs k-means from 10 different starts and keeps the best; `random_state=0` makes the result the same each time. Then it shows the mean of each column per cluster.

> **Predict.** Which column will separate the clusters most? Then run the cell.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

COLUMNS = ["tickets", "order_value", "word_count", "tenure_days"]
clusterer = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          KMeans(n_clusters=3, n_init=10, random_state=0))
customers["cluster"] = clusterer.fit_predict(customers[COLUMNS])
profile = customers.groupby("cluster")[COLUMNS].mean().round(1)
profile["customers"] = customers["cluster"].value_counts()
profile

> **Check.** You should see three clusters: cluster 0 with 374 customers and a mean tenure of 584.8 days, cluster 1 with 407 customers and 1858.2 days, and cluster 2 with 66 customers and 98.5 tickets on average.

Run the next cell. It counts the segments in each cluster, and the train tickets and escalation rate of each cluster. Clustering did not use the target; you look at it only now, to describe the groups.

In [ ]:
segment = known.groupby("customer_id")["segment"].first()
print(pd.crosstab(customers["cluster"], segment))
tickets = known.merge(customers["cluster"], left_on="customer_id", right_index=True)
print(tickets.groupby("cluster")[TARGET].agg(["size", "mean"]).round(3))

> **Check.** You should see that cluster 2 has 25 business, 1 home and 40 trade customers. The escalation rates are 0.102, 0.094 and 0.148.

## 4. A picture in two dimensions: PCA

Four columns cannot be drawn on a flat page. Like the shadow of a teapot on a wall, a picture keeps only part of the shape, so choose the angle that keeps the most. **PCA** (principal component analysis) makes new axes, each a weighted mix of the scaled columns, that keep as much of the spread as possible. The first two axes give a picture.

In [ ]:
from sklearn.decomposition import PCA

scaled = clusterer[:-1].transform(customers[COLUMNS])  # the fitted imputer and scaler
pca = PCA(n_components=2).fit(scaled)
print("share of the spread kept:", pca.explained_variance_ratio_.round(3))
print(pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=COLUMNS).round(2))

> **Check.** You should see `share of the spread kept: [0.309 0.267]`. PC1 has weights 0.75, 0.49, 0.12 and 0.42; PC2 has 0.03, −0.67, −0.0 and 0.74.

The two axes keep 57.5% of the spread, so the picture is useful but incomplete. Run the next cell to draw it.

In [ ]:
import matplotlib.pyplot as plt

xy = pca.transform(scaled)
fig, ax = plt.subplots(figsize=(6, 4.5))
for cluster in range(3):
    mine = customers["cluster"].to_numpy() == cluster
    ax.scatter(xy[mine, 0], xy[mine, 1], s=12, label=f"cluster {cluster}")
ax.set_xlabel("PC1 (more tickets, larger orders, longer tenure)")
ax.set_ylabel("PC2 (longer tenure, smaller orders)")
ax.legend()
plt.show()

> **Check.** You should see one plot with three colours. Cluster 2 spreads to the right (many tickets). Clusters 0 and 1 overlap at the left and differ mostly up and down (tenure).

The lesson page has the same picture for a fixed sample of 400 customers.

## 5. Choose k: hints, not answers

Two numbers help to choose k, the number of clusters:

- **Inertia**: the sum of the squared distances from each customer to its centre. It always falls when k grows.
- **Silhouette**: for each customer, how much closer it is to its own cluster than to the nearest other one, averaged, from −1 to 1. Near 0 means the groups overlap.

> **Predict.** Will one k be clearly the best? Then run the cell. It takes a few seconds.

In [ ]:
from sklearn.metrics import silhouette_score

for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(scaled)
    print(f"k {k}: inertia {km.inertia_:.0f}, silhouette {silhouette_score(scaled, km.labels_):.3f}")

> **Check.** You should see inertia fall from 2720 (k = 2) to 1134 (k = 8), and silhouette values from 0.231 to 0.285, highest at k = 5.

No k stands out: the inertia curve has no clear bend, and every silhouette is below 0.3, so the groups overlap. The numbers say "the customers do not fall into clear groups". Choose k by what people can use.

## 6. Guided practice: describe the clusters

> **Your turn.** Find the cluster with the highest mean `tickets` in `profile`, and put its number in `big`. Then put the share of the known customers' train tickets (the `tickets` table from section 3) that come from that cluster in `share_big`, rounded to 3 decimals.

In [ ]:
big = ...
share_big = ...
print(big, share_big)

In [ ]:
expect_hash('the share of the tickets from the busiest cluster', share_big, '97b95c97850b917d')

Write Grace's name for each cluster in the next cell (double-click to edit), and say if the cluster is useful to her. Then compare with the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/#grace-names-the-clusters).

*Your names here:*

| Cluster | Name | Useful? Why? |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |

## 7. Change one thing: k = 5

Silhouette was highest at k = 5. Change one setting and look.

> **Your turn.** Make the same pipeline with `n_clusters=5`. Put the cluster of each customer in `clusters5`, and the profile (mean of each column per cluster) in `profile5`. Then put the number of customers in the cluster with the highest mean `tickets` in `big5_customers`.

In [ ]:
clusters5 = ...
profile5 = ...
big5_customers = ...
profile5

In [ ]:
expect_hash('the customers in the busiest of five clusters', big5_customers, 'da4ea2a5506f2693')

Compare the five clusters with the three. Which cluster stays almost the same? What do the new clusters split on? The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/#your-turn-five-clusters) has the model answer.

## 8. Failure case: no scaling

Tomás leaves the scaler out. Run the next cell. It shows the smallest and largest `tickets` and `tenure_days` in each cluster.

> **Predict.** Which column will decide the clusters?

In [ ]:
unscaled = make_pipeline(SimpleImputer(strategy="median"), KMeans(n_clusters=3, n_init=10, random_state=0))
raw = customers.assign(cluster=unscaled.fit_predict(customers[COLUMNS]))
raw.groupby("cluster")[["tickets", "tenure_days"]].agg(["min", "max"])

> **Check.** You should see `tenure_days` in three bands that do not overlap: 2 to 868, 875 to 1737 and 1748 to 2607. The `tickets` ranges overlap completely.

Without scaling, `tenure_days` (SD 741) decides every distance, and `tickets` (SD 28) does not count. The clusters are three tenure bands that you could make with one line of pandas. The fix: scale first, in the same pipeline.

## 9. Module practice: misclassified tickets and clusters

Now combine the module. The next cell scores the validation month with the final model at 0.19, gives each ticket its customer's cluster (−1 for a guest or a customer who is new in May), and computes the recall per cluster.

> **Check.** You should see four rows: −1 with 24 tickets, and clusters 0, 1 and 2 with recall 0.5, 0.387 and 0.519.

In [ ]:
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
valid = load("valid")
valid["score"] = pipeline.predict_proba(valid[FEATURES])[:, 1]
valid["flag"] = (valid["score"] >= 0.19).astype(int)
valid["cluster"] = valid["customer_id"].map(customers["cluster"]).fillna(-1).astype(int)
valid["caught"] = valid["flag"] * valid[TARGET]
by_cluster = valid.groupby("cluster").agg(tickets=(TARGET, "size"), escalated=(TARGET, "sum"),
                                          caught=("caught", "sum"))
by_cluster["recall"] = (by_cluster["caught"] / by_cluster["escalated"]).round(3)
by_cluster

> **Your turn.** Inspect the missed escalations. In the next cell, select the escalated tickets that are not flagged and have a score below 0.10, and count them by `priority`, `team` and `cluster`. Then look at 10 of them.

In [ ]:
missed_low = ...
missed_low

In [ ]:
expect_hash('the number of missed escalations with a low score', len(missed_low) if hasattr(missed_low, 'columns') else missed_low, '02d20bbd7e394ad5')

Then write your findings in the next cell: **at least one data problem and one model limitation**, each with the numbers that show it. Use this lesson and the two before it. Compare with the model answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/#module-practice).

*Your findings here:*

- Data problem:
- Model limitation:
- Do the clusters show a useful pattern?

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you grouped 847 customers with k-means after scaling, and saw that without scaling the groups are only tenure bands. PCA kept 57.5% of the spread in a two-dimensional picture. Inertia and silhouette (all below 0.3) gave hints, not an answer. Grace found one useful group, 66 large accounts with 38.8% of the known customers' train tickets, and one split, newer against long-standing customers, that does not help her. Next, in Module 8, you package the model so that someone else can run it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.